ANALISI ESPLORATIVA

In [1]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from pydub import AudioSegment
from pathlib import Path
from datetime import datetime
import traceback
from scipy.signal import butter, sosfilt
import soundfile as sf
import pandas as pd
import os

In [2]:
# --- CONFIGURAZIONE PERCORSI ---
INPUT_DIR = Path("AudioSamplesExploration")
PREPROCESSED_DIR = Path("AudioSamplesExplorationPreprocessed")
WAVE_DIR = Path("WaveformsExploration")
SPECTRO_DIR = Path("SpectrogramsExploration")

# --- CONFIGURAZIONE RANGE ---
RANGES = {
    "mattina":("06:00", "12:00"),
    "pomeriggio": ("12:00", "19:00"),
    "sera": ("19:00", "23:59"),
    "notte": ("00:00", "06:00")
}
SAMPLE_RATE = 16000
FRAME_SIZE = 2048
HOP_LENGTH = 512
TARGET_DB = -1.0
target_amplitude = librosa.db_to_amplitude(TARGET_DB)
CUTOFF_FREQ = 100

def to_minutes(time_str):
    h, m = map(int, time_str.split(':'))
    return h * 60 + m

MINUTI_RANGES = {k: (to_minutes(v[0]), to_minutes(v[1])) for k, v in RANGES.items()}

DATE_FORMAT = "%Y-%m-%dT%H:%M:%S%z"
TITLE_DATE_FORMAT = "%d-%m-%Y, %H:%M:%S"

# Creazione automatica delle cartelle
for d in [WAVE_DIR, SPECTRO_DIR, PREPROCESSED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for fascia in RANGES.keys():
    # Creiamo le sottocartelle anche per i pre-processed
    for base in [WAVE_DIR, SPECTRO_DIR, PREPROCESSED_DIR]:
        (base / fascia).mkdir(parents=True, exist_ok=True)

In [3]:
def highpass_filter(data, cutoff, sr, order=5):
    # Nyquist frequency è la metà della frequenza di campionamento
    nyq = 0.5 * sr
    normal_cutoff = cutoff / nyq
    
    # Creiamo il filtro in formato SOS (Second-Order Sections) 
    # È più stabile numericamente rispetto al formato classico
    sos = butter(order, normal_cutoff, btype='high', analog=False, output='sos')
    
    # Applichiamo il filtro
    filtered_data = sosfilt(sos, data)
    return filtered_data

def preprocess(filename, timestamp_dt):
    try:
        
        # 1. ESTRAZIONE TIMESTAMP
        timestamp_str_title = timestamp_dt.strftime(TITLE_DATE_FORMAT)
        timestamp_str=timestamp_dt.strftime(DATE_FORMAT).replace(":", "-")[:-5]
        minuti_file = timestamp_dt.hour * 60 + timestamp_dt.minute
        
        fascia = "notte"
        for f, (inizio, fine) in MINUTI_RANGES.items():
            if inizio <= minuti_file < fine:
                fascia = f
                break

        # 2. CARICAMENTO E TAGLIO
        #audio = AudioSegment.from_wav(file_src)
        filepath = INPUT_DIR / filename
        audio, sr = librosa.load(filepath, sr=SAMPLE_RATE)
        offset = int(0.5 * sr)
        audio_cut = audio[offset:-offset]
        
        # 3. FILTRO E NORMALIZZAZIONE 
        #audio_clean = audio_cut.high_pass_filter(100, order=8)
        audio_no_dc = audio_cut - np.mean(audio_cut)
        audio_clean = highpass_filter(audio_no_dc, cutoff=CUTOFF_FREQ, sr=SAMPLE_RATE, order=8)
        #audio_normalized = audio_clean.normalize(headroom=1.0)
        audio_normalized = librosa.util.normalize(audio_clean) * target_amplitude

        # 4. SALVATAGGIO E LETTURA PER GRAFICI
        out_file_path = PREPROCESSED_DIR / fascia / f"preprocessed_audio_{timestamp_str}.wav"
        #audio_normalized.export(out_file_path, format="wav")
        sf.write(out_file_path, audio_normalized, SAMPLE_RATE)

        # --- GENERAZIONE GRAFICI ---
        # Waveform
        plt.figure(figsize=(10, 3))
        librosa.display.waveshow(audio_normalized, sr=SAMPLE_RATE, color="steelblue")
        plt.title(f"Waveform ({fascia}): {timestamp_str_title}")
        plt.tight_layout()
        plt.savefig(WAVE_DIR / fascia / f"waveform_{timestamp_str}.png")
        plt.close()

        # Spettrogramma
        plt.figure(figsize=(10, 4))
        S = librosa.feature.melspectrogram(y=audio_normalized, sr=SAMPLE_RATE, n_fft=FRAME_SIZE, hop_length=HOP_LENGTH, window='hann')
        S_dB = librosa.power_to_db(S, ref=np.max)
        img = librosa.display.specshow(S_dB, sr=SAMPLE_RATE, x_axis='time', y_axis='mel', cmap='magma')
        plt.colorbar(img, format='%+2.0f dB')
        plt.title(f"Mel-Spectrogram ({timestamp_str_title})")
        plt.tight_layout()
        plt.savefig(SPECTRO_DIR / fascia / f"spectrogram_{timestamp_str}.png")
        plt.close()

        print(f"Elaborato, salvato e smistato in {fascia}: {filename}")
    except Exception as e:
        traceback.print_exception(type(e), e, e.__traceback__)
        print(f"Errore su {filename}: {e}")

In [4]:
audio_df = pd.read_csv('audio_samples_metadata.csv')

files_in_folder = set(os.listdir("AudioSamplesExploration"))

for row in audio_df.itertuples():
    if row.filename in files_in_folder:
        timestamp=row.timestamp.replace(" ","T")
        timestamp_dt = datetime.strptime(timestamp, DATE_FORMAT)
        preprocess(row.filename, timestamp_dt)
    
print("Elaborazione completata.")

Elaborato, salvato e smistato in pomeriggio: audio_2026-03-09T16-34-22.wav
Elaborato, salvato e smistato in pomeriggio: audio_2026-03-09T16-51-54.wav
Elaborato, salvato e smistato in pomeriggio: audio_2026-03-09T17-43-06.wav
Elaborato, salvato e smistato in pomeriggio: audio_2026-03-09T17-58-27.wav
Elaborato, salvato e smistato in pomeriggio: audio_2026-03-09T18-18-55.wav
Elaborato, salvato e smistato in pomeriggio: audio_2026-03-09T18-29-08.wav
Elaborato, salvato e smistato in sera: audio_2026-03-09T19-15-07.wav
Elaborato, salvato e smistato in sera: audio_2026-03-09T19-40-40.wav
Elaborato, salvato e smistato in sera: audio_2026-03-09T20-31-46.wav
Elaborato, salvato e smistato in sera: audio_2026-03-09T20-41-59.wav
Elaborato, salvato e smistato in sera: audio_2026-03-09T21-17-45.wav
Elaborato, salvato e smistato in sera: audio_2026-03-09T21-38-12.wav
Elaborato, salvato e smistato in sera: audio_2026-03-09T22-39-32.wav
Elaborato, salvato e smistato in sera: audio_2026-03-09T22-59-58.wa